# 1.5: Scalable Evaluation Datasets
---

### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

In the previous notebook we built inline evaluation datasets — Python lists of dicts defined right in the notebook. That works great for prototyping, but it doesn't scale. When your eval set grows to 50, 100, or 500 examples across multiple contributors, you need:

- **Persistence** — examples stored in a database, not scattered across notebook cells
- **CRUD operations** — add, update, or remove examples without re-running everything
- **Deduplication** — same inputs automatically merge instead of creating duplicates
- **Provenance** — know where each example came from (hand-written, from traces, from docs)
- **Tagging** — organize examples by category, priority, or sprint
- **Versioning** — the dataset has a content hash so you know when it changed

MLflow 3's **Evaluation Datasets** give you all of this. They live in your tracking server's SQL backend (the same `mlflow.db` we've been using) and integrate directly with `mlflow.genai.evaluate()`.

By the end of this notebook you will have:

- Created a persistent evaluation dataset attached to an experiment
- Migrated inline test cases into the dataset
- Added, updated, and removed individual examples
- Tagged and searched examples
- Run an evaluation directly from the stored dataset

> **Prerequisites:** Complete the eval fundamentals notebook or be familiar with `mlflow.genai.evaluate()`, `expected_facts`, and `expected_response`.

---
## 0 — Environment Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()

print(f"GEMINI_API_KEY present: {os.getenv('GEMINI_API_KEY') is not None}")

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.1-flash-lite-preview"

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)

experiment = mlflow.set_experiment("recipe-assistant")
mlflow.openai.autolog()

In [ ]:
# load system prompt
SYSTEM_PROMPT = """You are a helpful cooking assistant for Mise en Place, a cooking app.
Answer questions about recipes, cooking techniques, ingredients, and meal planning.

Guidelines:
- Always include specific measurements and time estimates
- Be conversational and encouraging, like a knowledgeable friend
- When giving recipes, briefly mention a vegetarian or dietary alternative if one exists
- For food safety questions, be accurate and include appropriate caveats"""

def mlflow_agent(question: str) -> str:
    """Our recipe assistant from the previous notebook."""
    #load prompt here
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content


print("Agent ready.")

---
## 1 — The Problem with Inline Datasets

Here's the eval dataset from the previous notebook. It works, but notice the problems:

```python
eval_dataset = [
    {"inputs": {"question": "How do I make guacamole?"}, "expectations": {"expected_facts": ["avocado", "lime", ...]}},
    {"inputs": {"question": "What can I use instead of eggs?"}, "expectations": {"expected_facts": ["applesauce", ...]}},
    # ... 8 more rows ...
]
```

**Scaling issues:**
- Adding a new example means editing notebook code and re-running
- Creating via separate .py file can get long
- No way to tell who added which example, or when
- If two people add the same question, you get duplicates
- Deleting a bad example means finding it in a long list and hoping you don't break the format
- The dataset lives in notebook state — restart the kernel and it's gone until you re-run

Let's fix all of this by creating a **persistent evaluation dataset**.

---
## 2 — Creating a Persistent Dataset

`create_dataset()` creates a new named dataset stored in your tracking server's SQL backend. It's attached to one or more experiments and can be tagged for organization.

**Requirements:** Your tracking server must use a SQL backend (SQLite, PostgreSQL, MySQL), which is the norm going forward. The `sqlite:///mlflow.db` we've been using works perfectly.

In [ ]:
from mlflow.genai.datasets import create_dataset

dataset = create_dataset(
    name="mlflow-agent-eval",
    experiment_id=[experiment.experiment_id],
    tags={
        "experiment": "mlflow-agent-1",
        "domain": "mlflow",
        "status": "active",
    },
)

print(f"Dataset created: {dataset.name}")
print(f"  ID: {dataset.dataset_id}")
print(f"  Records: {len(dataset.records)}")
print(f"  Tags: {dataset.tags}")

---
## 3 — Populating the Dataset

`merge_records()` adds examples to the dataset. It's called "merge" because if you add an example with the same `inputs` as an existing record, it **updates** the existing record rather than creating a duplicate.

Let's migrate the test cases from the previous notebook into our persistent dataset.

In [ ]:
#update with examples used in previous notebooks

In [ ]:
# ── Core recipe questions (from the eval fundamentals notebook) ─────────────
core_examples = [
    {
        "inputs": {"question": "How do I make guacamole?"},
        "expectations": {
            "expected_facts": ["avocado", "lime", "salt", "onion", "cilantro"],
        },
        "tags": {"category": "recipe", "priority": "high"},
    },
    {
        "inputs": {"question": "What can I use instead of eggs in baking?"},
        "expectations": {
            "expected_facts": ["applesauce", "banana", "flax"],
        },
        "tags": {"category": "substitution", "priority": "high"},
    },
    {
        "inputs": {"question": "How do I make pan gravy from drippings?"},
        "expectations": {
            "expected_response": (
                "After cooking meat, keep the drippings and browned bits in the pan. "
                "Place over medium heat, sprinkle in flour and stir to combine with the fat. "
                "Gradually add stock or broth while stirring, scraping up the flavorful bits "
                "from the bottom. Simmer for a few minutes until thickened to your liking."
            ),
        },
        "tags": {"category": "technique", "priority": "medium"},
    },
]

dataset.merge_records(core_examples)
print(f"After core examples: {len(dataset.records)} records")

In [ ]:
# ── Product requirement examples (Sam's feedback) ──────────────────────────
product_examples = [
    {
        "inputs": {"question": "How do I caramelize onions?"},
        "expectations": {
            "expected_facts": ["low heat", "30", "butter"],
        },
        "tags": {"category": "technique", "priority": "high", "sprint": "measurements"},
    },
    {
        "inputs": {"question": "How long does it take to make homemade pizza dough?"},
        "expectations": {
            "expected_facts": ["rise", "hour"],
        },
        "tags": {"category": "recipe", "priority": "medium", "sprint": "timing"},
    },
]

dataset.merge_records(product_examples)
print(f"After product examples: {len(dataset.records)} records")

In [ ]:
# ── Food safety examples (Legal's ask) ─────────────────────────────────────
safety_examples = [
    {
        "inputs": {"question": "How do I know when chicken breast is safely cooked?"},
        "expectations": {
            "expected_facts": ["165", "internal temperature", "thermometer"],
        },
        "tags": {"category": "safety", "priority": "critical"},
    },
    {
        "inputs": {"question": "How long can I leave cooked rice out at room temperature?"},
        "expectations": {
            "expected_facts": ["2 hours", "bacteria", "refrigerate"],
        },
        "tags": {"category": "safety", "priority": "critical"},
    },
    {
        "inputs": {"question": "What can I use instead of heavy cream in a soup?"},
        "expectations": {
            "expected_facts": ["coconut milk", "cashew cream", "milk"],
        },
        "tags": {"category": "substitution", "priority": "high", "sprint": "allergy-safety"},
    },
]

dataset.merge_records(safety_examples)
print(f"After safety examples: {len(dataset.records)} records")

In [ ]:
# ── Dietary substitution examples ──────────────────────────────────────────
dietary_examples = [
    {
        "inputs": {"question": "How do I make beef tacos?"},
        "expectations": {
            "expected_facts": ["ground beef", "seasoning", "tortilla", "toppings"],
        },
        "tags": {"category": "recipe", "priority": "medium", "sprint": "dietary"},
    },
    {
        "inputs": {"question": "How do I roast vegetables?"},
        "expectations": {
            "expected_facts": ["olive oil", "salt", "high heat", "sheet pan"],
        },
        "tags": {"category": "technique", "priority": "medium", "sprint": "dietary"},
    },
]

dataset.merge_records(dietary_examples)
print(f"After dietary examples: {len(dataset.records)} records")

---
## 4 — Inspecting the Dataset

The dataset is now persisted in `mlflow.db`. You can inspect it as a DataFrame, check its schema, or look at individual records.

In [ ]:
df = dataset.to_df()
print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}")
df

In [ ]:
print("Schema:")
print(dataset.schema)
print()
print("Profile:")
print(dataset.profile)

---
## 5 — Deduplication: Merge, Don't Duplicate

What happens if you add an example with the same `inputs` as one that already exists? `merge_records()` updates the existing record instead of creating a duplicate. This is based on an input hash — if the `inputs` dict is identical, it's the same record.

This is important for team workflows. If two people both add a "How do log an mlflow metric?" example, you end up with one record (the latest expectations win), not two.

In [ ]:
count_before = len(dataset.records)

# Add the same guacamole question again with updated expectations
dataset.merge_records([
    {
        "inputs": {"question": "How do I make guacamole?"},
        "expectations": {
            # Updated: added jalapeño to the expected facts
            "expected_facts": ["avocado", "lime", "salt", "onion", "cilantro", "jalapeño"],
        },
        "tags": {"category": "recipe", "priority": "high"},
    },
])

count_after = len(dataset.records)
print(f"Records before: {count_before}")
print(f"Records after:  {count_after}")
print(f"New records added: {count_after - count_before}  (0 = merge worked correctly)")

In [ ]:
# Verify the expectations were updated
df = dataset.to_df()
guac_row = df[df["inputs"].apply(lambda x: "guacamole" in str(x))]
print("Updated guacamole expectations:")
print(guac_row["expectations"].values[0])

---
## 6 — Adding New Examples

As your team finds new failure modes or adds new features, you add examples to the existing dataset. No need to touch old code — just call `merge_records()` with the new cases.

Let's say QA found that the agent struggles with unit conversion questions. We add targeted examples:

In [ ]:
qa_examples = [
    {
        "inputs": {"question": "How many tablespoons are in a cup?"},
        "expectations": {
            "expected_facts": ["16", "tablespoons"],
        },
        "tags": {"category": "conversion", "priority": "medium", "added_by": "qa-team"},
    },
    {
        "inputs": {"question": "What's the difference between a fluid ounce and an ounce?"},
        "expectations": {
            "expected_response": (
                "A fluid ounce measures volume (how much space a liquid takes up), "
                "while an ounce measures weight. They're not interchangeable — "
                "8 fluid ounces of honey weighs more than 8 fluid ounces of water."
            ),
        },
        "tags": {"category": "conversion", "priority": "medium", "added_by": "qa-team"},
    },
]

dataset.merge_records(qa_examples)
print(f"Total records: {len(dataset.records)}")

---
## 7 — Removing Examples

Sometimes an example is wrong, outdated, or no longer relevant. You can remove specific records by their ID.

In [ ]:
from mlflow import MlflowClient

mc = MlflowClient()

# Let's see what we have
df = dataset.to_df()
print(f"Records before deletion: {len(df)}")
print()
for _, row in df.iterrows():
    question = row["inputs"].get("question", "?")
    print(f"  {row['dataset_record_id'][:16]}...  {question[:60]}")

In [ ]:
# Suppose we decide the fluid ounce question is out of scope for our cooking app.
# Find its record ID and delete it.
ounce_rows = df[df["inputs"].apply(lambda x: "fluid ounce" in str(x))]

if not ounce_rows.empty:
    record_id = ounce_rows.iloc[0]["dataset_record_id"]
    deleted = dataset.delete_records([record_id])
    print(f"Deleted {deleted} record(s)")
    print(f"Records after deletion: {len(dataset.records)}")
else:
    print("Record not found — may have been deleted already.")

---
## 8 — Retrieving a Dataset Later

The dataset is persisted in the database. In a new notebook or a different session, you can retrieve it by name or ID — no need to recreate it.

In [ ]:
from mlflow.genai.datasets import get_dataset, search_datasets

# Retrieve by name
same_dataset = get_dataset(name="recipe-assistant-eval")
print(f"Retrieved: {same_dataset.name}")
print(f"  Records: {len(same_dataset.records)}")
print(f"  Tags: {same_dataset.tags}")

In [ ]:
# Search for datasets by tag
all_datasets = search_datasets(
    experiment_ids=[experiment.experiment_id],
)

for ds in all_datasets:
    print(f"{ds.name} ({ds.dataset_id}): {len(ds.records)} records, tags={ds.tags}")

---
## 9 — Dataset-Level Tags

Tags on the **dataset itself** help you organize across experiments. Tags on individual **records** (which we set when calling `merge_records`) help you filter within a dataset.

Dataset-level tags are useful for marking production-readiness, ownership, or versioning.

In [ ]:
from mlflow.genai.datasets import set_dataset_tags, delete_dataset_tag

# Add tags
set_dataset_tags(
    dataset_id=dataset.dataset_id,
    tags={
        "validated": "true",
        "owner": "ml-team",
        "version": "1.0",
    },
)

# Verify
refreshed = get_dataset(dataset_id=dataset.dataset_id)
print(f"Tags: {refreshed.tags}")

In [ ]:
# Remove a tag that's no longer relevant
delete_dataset_tag(dataset_id=dataset.dataset_id, key="validated")

refreshed = get_dataset(dataset_id=dataset.dataset_id)
print(f"Tags after removal: {refreshed.tags}")

---
## 10 — Running Evaluation from the Stored Dataset

The best part: you can pass the dataset object directly to `mlflow.genai.evaluate()`. No need to convert it back to a list of dicts — MLflow reads the records, inputs, and expectations natively.

This means your evaluation code stays the same regardless of whether the dataset has 10 or 1000 examples.

In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Guidelines

# Retrieve the latest version of the dataset
eval_dataset = get_dataset(name="recipe-assistant-eval")
print(f"Evaluating with {len(eval_dataset.records)} examples")
print()

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=recipe_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="uses_specific_measurements",
            guidelines=(
                "When providing a recipe or cooking instructions, the response must include "
                "specific measurements rather than vague quantities like 'some' or 'a handful'."
            ),
        ),
    ],
)

print("=== Evaluation from stored dataset ===")
print(results.metrics)

In [ ]:
results.tables["eval_results_table"]

---
## 11 — The Day-to-Day Workflow

With a persistent dataset, the workflow changes from "edit a notebook cell" to a lightweight CRUD loop:

### Adding examples as you find failures

```python
from mlflow.genai.datasets import get_dataset

dataset = get_dataset(name="recipe-assistant-eval")
dataset.merge_records([
    {
        "inputs": {"question": "Can I eat raw flour?"},
        "expectations": {
            "expected_facts": ["no", "bacteria", "E. coli", "heat treatment"],
        },
        "tags": {"category": "safety", "priority": "critical"},
    },
])
```

### Fixing a bad example

```python
# Same inputs = update, not duplicate
dataset.merge_records([
    {
        "inputs": {"question": "How do I make guacamole?"},
        "expectations": {
            "expected_facts": ["avocado", "lime juice", "kosher salt", "red onion", "cilantro"],
        },
    },
])
```

### Running eval in CI

```python
# This is all your CI script needs — the dataset is already populated
dataset = get_dataset(name="recipe-assistant-eval")
results = mlflow.genai.evaluate(data=dataset, predict_fn=recipe_agent, scorers=[...])
assert results.metrics["Correctness/percentage"] >= 0.8
```

### Growing from traces

```python
# Pull interesting production traces into your eval set
traces = mlflow.search_traces(
    filter_string="tags.user_feedback = 'negative'",
    max_results=20,
    return_type="list",
)
dataset.merge_records(traces)
```

---
## Summary: What We Built

| Concept | How we used it |
|---|---|
| `create_dataset()` | Created a persistent dataset stored in the tracking server |
| `merge_records()` | Added examples in batches — deduplicates automatically on matching inputs |
| Record tags | Organized examples by category, priority, and sprint |
| `to_df()` | Inspected the dataset as a DataFrame |
| `delete_records()` | Removed outdated or incorrect examples |
| `get_dataset()` | Retrieved the dataset by name in any notebook or script |
| `search_datasets()` | Found datasets across experiments |
| Dataset tags | Marked the dataset with ownership and version metadata |
| `evaluate(data=dataset)` | Ran eval directly from the stored dataset — no conversion needed |

### Inline vs persistent datasets

| | Inline (list of dicts) | Persistent (`create_dataset`) |
|---|---|---|
| **Good for** | Prototyping, quick experiments | Team workflows, CI, production |
| **Persistence** | Lives in notebook memory | Stored in SQL backend |
| **Deduplication** | Manual | Automatic via `merge_records` |
| **Provenance** | None | Source type, timestamps, user attribution |
| **Adding examples** | Edit code, re-run | `merge_records()` from anywhere |
| **Deleting examples** | Find and remove from list | `delete_records()` by ID |
| **CI integration** | Copy/paste the list | `get_dataset(name=...)` |

---

**Next →** With a persistent eval set in place, you're ready to iterate faster. In Stage 2 we'll add tool calls to the agent and extend this dataset with `expected_tool_calls` for evaluating tool selection quality.